In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockA_hybrid_O3
# Build hybrid-level O3 partial-column caches from raw h3 files
#
# Data source:
#   - Climatology: B2000WCN (raw hybrid h3 files)
#   - Reference   : BWCN      (raw hybrid h3 files)
#
# Outputs for each pressure range:
#   1) full B2000 daily TS
#   2) B2000 Oct-Sep-by-year matrix
#   3) lowest10 / lowest25 year txt
#   4) all-year Oct-Sep climatology mean/std
#   5) low25    Oct-Sep composite  mean/std
#   6) BWCN Oct-Sep reference series for selected years
# ============================================================

import os
import re
import glob
import gc

import numpy as np
import xarray as xr
from tqdm import tqdm

# ============================================================
# CONFIG
# ============================================================

DATA_ROOT = "/mnt/soclim0/public_data/weiji"

BWCN_ROOT = os.path.join(DATA_ROOT, "BWCN")
B2000_ROOT = os.path.join(DATA_ROOT, "B2000WCN001002")

ROOT_DIR = "/home/weiji/restart_exam/code/20260415egu/plots/General_O3_hybrid"
os.makedirs(ROOT_DIR, exist_ok=True)

OPEN_ENGINE = "netcdf4"
USE_CACHE = True

# Oct-Dec previous year + Jan-Sep current year
N_PREV = 91
N_DAYS = 365

PRESSURE_RANGES = [
    (30.0, 70.0, "30_70hPa"),
    (1.0, 100.0, "1_100hPa"),
]

# BWCN years to overlay in fig2
BWCN_SPECIAL_YEARS = [8, 13, 14, 19]

# ============================================================
# HELPERS
# ============================================================

def parse_b2000_file_year(path):
    base = os.path.basename(path)
    m = re.search(r"\.cam\.h3\.(\d{4})\.", base)
    if not m:
        raise ValueError(f"Cannot parse B2000 file year from: {base}")
    return int(m.group(1))

def get_b2000_files(varname="O3"):
    root = os.path.join(B2000_ROOT, varname)
    files = sorted(glob.glob(os.path.join(root, f"B2000WCN.sample.cam.h3.*.{varname}.nc")))
    if not files:
        raise FileNotFoundError(f"No B2000WCN {varname} files found under {root}")
    return files

def find_bwcn_var_files_for_year(year_str, varname="O3"):
    root = os.path.join(BWCN_ROOT, varname)
    patterns = [
        os.path.join(root, f"BWCN.cam.h3.{year_str}.{varname}.nc"),
        os.path.join(root, f"BWCN.sample.cam.h3.{year_str}.{varname}.nc"),
        os.path.join(root, f"*{year_str}*.{varname}.nc"),
    ]
    for pat in patterns:
        files = sorted(glob.glob(pat))
        if files:
            return files
    raise FileNotFoundError(
        f"No BWCN {varname} files found for year {year_str} under {root}"
    )

def trim_dataset_for_o3(ds):
    keep = {"O3", "PS", "P0", "hyai", "hybi", "gw", "time", "lat", "lon", "lev", "ilev"}
    drop_vars = [v for v in ds.variables if v not in keep]
    if drop_vars:
        ds = ds.drop_vars(drop_vars)
    return ds

def get_lat_weights(ds_or_da, lat_slice):
    if "gw" in ds_or_da:
        return ds_or_da["gw"].sel(lat=lat_slice)
    lat = ds_or_da["lat"].sel(lat=lat_slice)
    return xr.DataArray(np.cos(np.deg2rad(lat)), dims=["lat"], coords={"lat": lat})

def shift_years_of_time_coord(time_values, offset):
    out = []
    for t in time_values:
        if not hasattr(t, "replace"):
            raise TypeError(
                "Time coordinate does not support .replace(year=...). "
                "Expected cftime-like time objects in raw model output."
            )
        out.append(t.replace(year=int(t.year) + int(offset)))
    return out

# ============================================================
# HYBRID O3 CORE
# ============================================================

def calc_parc_o3_hybrid(ds_sub, p_top_hpa, p_bot_hpa, verbose=False):
    """
    Partial O3 column on CAM/WACCM hybrid levels using interface-overlap dp.
    O3 is assumed to be mole fraction (mol/mol).
    Output: DU
    """
    g = 9.80665
    M_air = 28.964 / 1000.0
    Na = 6.02214e23
    DU = 2.687e20
    factor = Na / (g * M_air * DU)

    P0 = ds_sub["P0"].squeeze(drop=True)
    PS = ds_sub["PS"]

    # interface pressure (Pa), dims typically: time, ilev, lat, lon
    P_interface = ds_sub["hyai"] * P0 + ds_sub["hybi"] * PS

    p_i   = P_interface.isel(ilev=slice(0, -1)).rename({"ilev": "lev"})
    p_ip1 = P_interface.isel(ilev=slice(1, None)).rename({"ilev": "lev"})

    if "lev" in ds_sub.coords:
        lev_vals = ds_sub["lev"].values
        p_i = p_i.assign_coords(lev=lev_vals)
        p_ip1 = p_ip1.assign_coords(lev=lev_vals)

    p_layer_top = xr.where(p_i < p_ip1, p_i, p_ip1)
    p_layer_bot = xr.where(p_i > p_ip1, p_i, p_ip1)

    pT = p_top_hpa * 100.0
    pB = p_bot_hpa * 100.0

    upper = xr.where(p_layer_top > pT, p_layer_top, pT)
    lower = xr.where(p_layer_bot < pB, p_layer_bot, pB)
    overlap = xr.where(lower > upper, lower - upper, 0.0)

    if "lev" in ds_sub.coords:
        overlap = overlap.assign_coords(lev=ds_sub["lev"].values)

    O3_col = (ds_sub["O3"] * overlap * factor).sum(dim="lev")

    if verbose:
        dp_eff = overlap.sum("lev") / 100.0
        print(f"[Hybrid O3] {p_top_hpa:.1f}-{p_bot_hpa:.1f} hPa")
        print("  Effective dp global mean (hPa):", float(dp_eff.mean().load()))

    return O3_col

def compute_o3_pc_ts(ds, p_top_hpa, p_bot_hpa):
    lat_slice = slice(60, 90)

    ds_sub = ds.sel(lat=lat_slice).load()

    O3_pc_3d = calc_parc_o3_hybrid(ds_sub, p_top_hpa, p_bot_hpa, verbose=False)
    O3_zm = O3_pc_3d.mean(dim="lon")
    weights = get_lat_weights(ds_sub, lat_slice)
    O3_ts = O3_zm.weighted(weights).mean(dim="lat")

    O3_ts.name = f"O3_pc_{int(p_top_hpa)}_{int(p_bot_hpa)}hPa_60_90N"
    return O3_ts

# ============================================================
# CROSS-YEAR HELPERS
# ============================================================

def build_octsep_sequence(ts_1d, target_year, n_prev=91, n_days=365):
    """
    Build one Oct(target_year-1) -> Sep(target_year) sequence.
    """
    prev = ts_1d.where(ts_1d.time.dt.year == (target_year - 1), drop=True)
    cur  = ts_1d.where(ts_1d.time.dt.year == target_year, drop=True)

    if prev.size != n_days or cur.size != n_days:
        return None

    seq = np.concatenate([
        np.asarray(prev)[n_days - n_prev:n_days],  # Oct-Dec
        np.asarray(cur)[0:n_days - n_prev],        # Jan-Sep
    ])

    if seq.size != n_days:
        return None

    return seq

def build_octsep_matrix_from_ts(ts_1d, n_prev=91, n_days=365):
    years = np.unique(ts_1d.time.dt.year.values.astype(int))

    seqs = []
    valid_years = []

    for y in years:
        seq = build_octsep_sequence(ts_1d, y, n_prev=n_prev, n_days=n_days)
        if seq is None:
            continue
        seqs.append(seq)
        valid_years.append(int(y))

    if len(seqs) == 0:
        raise RuntimeError("No valid Oct-Sep sequences could be built.")

    da = xr.DataArray(
        np.asarray(seqs, dtype=np.float64),
        dims=["year", "season_day"],
        coords={
            "year": np.asarray(valid_years, dtype=int),
            "season_day": np.arange(n_days, dtype=int),
        },
        name="O3_pc_OctSep_by_year",
    )
    return da

def select_low_years_from_spring_min(ts_1d, valid_years):
    """
    Low years are defined from Mar-Apr daily minimum in the CURRENT year.
    Restrict to years that have valid Oct-Sep sequences.
    """
    spring = ts_1d.where(ts_1d.time.dt.month.isin([3, 4]), drop=True)
    spring = spring.where(spring.time.dt.year.isin(valid_years), drop=True)

    spring_min_by_year = spring.groupby("time.year").min("time")

    spring_years = spring_min_by_year["year"].values.astype(int)
    spring_vals = spring_min_by_year.values

    idx_sorted = np.argsort(spring_vals)

    n_low10 = min(10, len(spring_years))
    lowest10_years = spring_years[idx_sorted[:n_low10]]

    n_low25 = max(int(0.25 * len(spring_years)), 1)
    lowest25_years = spring_years[idx_sorted[:n_low25]]

    return lowest10_years, lowest25_years

# ============================================================
# B2000 BUILDERS
# ============================================================

def build_b2000_o3_timeseries(p_top_hpa, p_bot_hpa, tag, use_cache=True):
    out_nc = os.path.join(ROOT_DIR, f"O3_pc_B2000_60_90N_{tag}_ts.nc")

    if use_cache and os.path.exists(out_nc):
        da = xr.open_dataarray(out_nc)
        da.load()
        return da

    all_files = get_b2000_files("O3")

    valid_files = []
    for f in all_files:
        yr = parse_b2000_file_year(f)
        if (1 <= yr <= 103) or (105 <= yr <= 210):
            valid_files.append((yr, f))
    valid_files.sort(key=lambda x: x[0])

    ts_list = []

    print(f"[B2000] Building O3 TS for {tag} year-by-year ...")
    for file_year, f in tqdm(valid_files, desc=f"B2000 {tag}"):
        with xr.open_dataset(f, engine=OPEN_ENGINE, cache=False) as ds:
            ds = trim_dataset_for_o3(ds)

            # Match your existing B2000 logic exactly
            if file_year >= 105:
                new_times = shift_years_of_time_coord(ds.time.values, 103)
                ds = ds.assign_coords(time=new_times)

            ts_yr = compute_o3_pc_ts(ds, p_top_hpa, p_bot_hpa)
            ts_yr = ts_yr.load()
            ts_list.append(ts_yr)

        gc.collect()

    ts = xr.concat(ts_list, dim="time").sortby("time")
    ts.name = f"O3_pc_B2000_60_90N_{tag}"
    ts.to_netcdf(out_nc)

    print(f"[B2000] Saved TS: {out_nc}")
    return ts

def build_b2000_octsep_and_climatology(ts_b2000, tag, n_prev=91, n_days=365):
    octsep_nc = os.path.join(ROOT_DIR, f"O3_pc_B2000_60_90N_{tag}_OctSep_by_year.nc")

    low10_txt = os.path.join(ROOT_DIR, f"O3_lowest10_years_B2000_{tag}.txt")
    low25_txt = os.path.join(ROOT_DIR, f"O3_lowest25pct_years_B2000_{tag}.txt")

    all_mean_nc = os.path.join(ROOT_DIR, f"O3_pc_B2000_60_90N_{tag}_OctSep_all_mean.nc")
    all_std_nc  = os.path.join(ROOT_DIR, f"O3_pc_B2000_60_90N_{tag}_OctSep_all_std.nc")

    low25_mean_nc = os.path.join(ROOT_DIR, f"O3_pc_B2000_60_90N_{tag}_OctSep_low25_mean.nc")
    low25_std_nc  = os.path.join(ROOT_DIR, f"O3_pc_B2000_60_90N_{tag}_OctSep_low25_std.nc")

    if (
        USE_CACHE
        and os.path.exists(octsep_nc)
        and os.path.exists(low10_txt)
        and os.path.exists(low25_txt)
        and os.path.exists(all_mean_nc)
        and os.path.exists(all_std_nc)
        and os.path.exists(low25_mean_nc)
        and os.path.exists(low25_std_nc)
    ):
        print(f"[B2000] Using cached Oct-Sep + climatology files for {tag}")
        return

    octsep_da = build_octsep_matrix_from_ts(ts_b2000, n_prev=n_prev, n_days=n_days)
    valid_years = octsep_da.year.values.astype(int)

    lowest10_years, lowest25_years = select_low_years_from_spring_min(ts_b2000, valid_years)

    np.savetxt(low10_txt, lowest10_years, fmt="%04d")
    np.savetxt(low25_txt, lowest25_years, fmt="%04d")

    all_mean = octsep_da.mean("year", skipna=True)
    all_std  = octsep_da.std("year", skipna=True)

    low25_da = octsep_da.sel(year=lowest25_years)
    low25_mean = low25_da.mean("year", skipna=True)
    low25_std  = low25_da.std("year", skipna=True)

    octsep_da.name = f"O3_pc_B2000_60_90N_{tag}_OctSep_by_year"
    all_mean.name = f"O3_pc_B2000_60_90N_{tag}_OctSep_all_mean"
    all_std.name = f"O3_pc_B2000_60_90N_{tag}_OctSep_all_std"
    low25_mean.name = f"O3_pc_B2000_60_90N_{tag}_OctSep_low25_mean"
    low25_std.name = f"O3_pc_B2000_60_90N_{tag}_OctSep_low25_std"

    octsep_da.to_netcdf(octsep_nc)
    all_mean.to_netcdf(all_mean_nc)
    all_std.to_netcdf(all_std_nc)
    low25_mean.to_netcdf(low25_mean_nc)
    low25_std.to_netcdf(low25_std_nc)

    print(f"[B2000] Saved Oct-Sep matrix: {octsep_nc}")
    print(f"[B2000] Saved lowest10 years:  {low10_txt}")
    print(f"[B2000] Saved lowest25 years: {low25_txt}")
    print(f"[B2000] Saved all-year climatology mean/std for {tag}")
    print(f"[B2000] Saved low25 composite mean/std for {tag}")

# ============================================================
# BWCN BUILDERS
# ============================================================

def build_bwcn_single_year_ts(year_int, p_top_hpa, p_bot_hpa, tag, use_cache=True):
    year_str = f"{year_int:04d}"
    out_nc = os.path.join(ROOT_DIR, f"O3_pc_BWCN_60_90N_{tag}_year{year_str}.nc")

    if use_cache and os.path.exists(out_nc):
        da = xr.open_dataarray(out_nc)
        da.load()
        return da

    files = find_bwcn_var_files_for_year(year_str, "O3")

    ts_parts = []

    for f in files:
        with xr.open_dataset(f, engine=OPEN_ENGINE, cache=False) as ds:
            ds = trim_dataset_for_o3(ds)
            ts_part = compute_o3_pc_ts(ds, p_top_hpa, p_bot_hpa)
            ts_part = ts_part.load()
            ts_parts.append(ts_part)

        gc.collect()

    if len(ts_parts) == 1:
        ts = ts_parts[0]
    else:
        ts = xr.concat(ts_parts, dim="time").sortby("time")

    ts.name = f"O3_pc_BWCN_60_90N_{tag}_year{year_str}"
    ts.to_netcdf(out_nc)

    print(f"[BWCN] Saved single-year TS: {out_nc}")
    return ts

def build_bwcn_octsep_series(target_year, p_top_hpa, p_bot_hpa, tag, n_prev=91, n_days=365):
    year_str = f"{target_year:04d}"
    out_nc = os.path.join(ROOT_DIR, f"O3_pc_BWCN_60_90N_{tag}_OctSep_year{year_str}.nc")

    if USE_CACHE and os.path.exists(out_nc):
        da = xr.open_dataarray(out_nc)
        da.load()
        return da

    ts_prev = build_bwcn_single_year_ts(target_year - 1, p_top_hpa, p_bot_hpa, tag, use_cache=USE_CACHE)
    ts_cur  = build_bwcn_single_year_ts(target_year,     p_top_hpa, p_bot_hpa, tag, use_cache=USE_CACHE)

    if ts_prev.size != n_days or ts_cur.size != n_days:
        raise RuntimeError(
            f"BWCN year {target_year:04d}: previous/current year does not have {n_days} days."
        )

    seq = np.concatenate([
        np.asarray(ts_prev)[n_days - n_prev:n_days],
        np.asarray(ts_cur)[0:n_days - n_prev],
    ])

    da = xr.DataArray(
        seq,
        dims=["season_day"],
        coords={"season_day": np.arange(n_days, dtype=int)},
        name=f"O3_pc_BWCN_60_90N_{tag}_OctSep_year{year_str}",
    )
    da.attrs["target_year"] = int(target_year)
    da.to_netcdf(out_nc)

    print(f"[BWCN] Saved Oct-Sep series: {out_nc}")
    return da

# ============================================================
# RUN
# ============================================================

for ptop, pbot, tag in PRESSURE_RANGES:
    print("\n" + "=" * 80)
    print(f"[RUN] Processing hybrid O3 partial column for {ptop:.1f}-{pbot:.1f} hPa ({tag})")
    print("=" * 80)

    # 1) B2000 daily TS
    ts_b2000 = build_b2000_o3_timeseries(ptop, pbot, tag, use_cache=USE_CACHE)

    # 2) B2000 Oct-Sep matrix + low10/low25 + climatology
    build_b2000_octsep_and_climatology(
        ts_b2000=ts_b2000,
        tag=tag,
        n_prev=N_PREV,
        n_days=N_DAYS,
    )

    # 3) BWCN Oct-Sep reference series
    for y in BWCN_SPECIAL_YEARS:
        try:
            build_bwcn_octsep_series(
                target_year=y,
                p_top_hpa=ptop,
                p_bot_hpa=pbot,
                tag=tag,
                n_prev=N_PREV,
                n_days=N_DAYS,
            )
        except Exception as e:
            print(f"[WARN] Failed BWCN year {y:04d} for {tag}: {e}")

print("\n[Done] All hybrid O3 caches generated successfully.")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockB_hybrid_O3_plot
# Plot from hybrid-level raw h3 caches
#
# Fig1:
#   lowest10 years vs all-other-years climatology ± std
#
# Fig2:
#   BWCN special years vs B2000 all-year climatology
#   and B2000 low25 composite
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from matplotlib.pyplot import cm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib import rcParams

# ============================================================
# CONFIG
# ============================================================

ROOT_DIR = "/home/weiji/restart_exam/code/20260415egu/plots/General_O3_hybrid"
os.makedirs(ROOT_DIR, exist_ok=True)

N_PREV = 91
N_DAYS = 365

PRESSURE_RANGES = [
    (30.0, 70.0, "30_70hPa"),
    (1.0, 100.0, "1_100hPa"),
]

BWCN_SPECIAL_YEARS = [8, 13, 14, 19]

# ============================================================
# HELPERS
# ============================================================

def load_da(path):
    da = xr.open_dataarray(path)
    da.load()
    return da

def load_year_txt(path):
    arr = np.loadtxt(path, dtype=int)
    return np.atleast_1d(arr).astype(int)

x_full = np.arange(N_DAYS)

month_ticks = [0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334]
month_labels = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "June", "July", "Aug", "Sep"]

# ============================================================
# RUN PLOTS
# ============================================================

for ptop, pbot, tag in PRESSURE_RANGES:
    print(f"\n[Plot] ===== {ptop:.1f}-{pbot:.1f} hPa ({tag}) =====")

    # --------------------------------------------------------
    # Load B2000 caches
    # --------------------------------------------------------
    octsep_nc = os.path.join(ROOT_DIR, f"O3_pc_B2000_60_90N_{tag}_OctSep_by_year.nc")
    low10_txt = os.path.join(ROOT_DIR, f"O3_lowest10_years_B2000_{tag}.txt")
    low25_txt = os.path.join(ROOT_DIR, f"O3_lowest25pct_years_B2000_{tag}.txt")

    all_mean_nc = os.path.join(ROOT_DIR, f"O3_pc_B2000_60_90N_{tag}_OctSep_all_mean.nc")
    all_std_nc  = os.path.join(ROOT_DIR, f"O3_pc_B2000_60_90N_{tag}_OctSep_all_std.nc")

    low25_mean_nc = os.path.join(ROOT_DIR, f"O3_pc_B2000_60_90N_{tag}_OctSep_low25_mean.nc")
    low25_std_nc  = os.path.join(ROOT_DIR, f"O3_pc_B2000_60_90N_{tag}_OctSep_low25_std.nc")

    octsep_da = load_da(octsep_nc)
    lowest10_years = load_year_txt(low10_txt)
    lowest25_years = load_year_txt(low25_txt)

    all_mean = load_da(all_mean_nc)
    all_std  = load_da(all_std_nc)
    low25_mean = load_da(low25_mean_nc)
    low25_std  = load_da(low25_std_nc)

    years_b2000 = octsep_da.year.values.astype(int)

    print("[Plot] B2000 valid years:", years_b2000)
    print("[Plot] lowest10 years:", lowest10_years)
    print("[Plot] lowest25 years:", lowest25_years)

    # ========================================================
    # FIG 1: lowest10 vs other years
    # ========================================================
    fig1, ax1 = plt.subplots(1, 1, figsize=(13, 10), constrained_layout=True)

    low_years = lowest10_years
    neutral_years = years_b2000[~np.isin(years_b2000, low_years)]

    low_da = octsep_da.sel(year=low_years)
    neu_da = octsep_da.sel(year=neutral_years)

    neu_mean = neu_da.mean("year", skipna=True).values
    neu_std  = neu_da.std("year", skipna=True).values

    colors_ext = cm.twilight(np.linspace(0, 1, len(low_years)))

    for j, y in enumerate(low_years):
        ax1.plot(
            x_full,
            low_da.sel(year=y).values,
            color=colors_ext[j],
            alpha=0.8,
            linewidth=2,
            label="low O3 years" if j == 0 else None,
        )

    ax1.errorbar(
        x_full,
        neu_mean,
        neu_std,
        color="grey",
        alpha=0.5,
        elinewidth=1.5,
        linewidth=3,
        label="all other years",
    )

    ax1.set_xlim(0, N_DAYS - 1)
    ax1.set_xticks(month_ticks)
    ax1.set_xticklabels(month_labels, fontsize=15)
    ax1.set_ylabel(f"Partial ozone column, {int(ptop)}–{int(pbot)} hPa (DU)", fontsize=18)
    ax1.tick_params(axis="y", labelsize=18)
    ax1.legend(fontsize=14)

    out_png1 = os.path.join(ROOT_DIR, f"O3_evolution_min_10_B2000_hybrid_{tag}.png")
    out_pdf1 = os.path.join(ROOT_DIR, f"O3_evolution_min_10_B2000_hybrid_{tag}.pdf")

    plt.savefig(out_png1, dpi=300)
    plt.savefig(out_pdf1)
    plt.show()

    print("[Plot] Saved fig1:")
    print("   ", out_png1)
    print("   ", out_pdf1)

    # ========================================================
    # FIG 2: BWCN special years vs B2000 climatology & low25
    # ========================================================
    rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans"],
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 10,
        "axes.linewidth": 0.8,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    })

    fig2, ax2 = plt.subplots(1, 1, figsize=(6.5, 4.0), constrained_layout=True)

    colors_spec = plt.cm.tab10(np.linspace(0, 1, len(BWCN_SPECIAL_YEARS)))
    handles_years = []

    for i, y in enumerate(BWCN_SPECIAL_YEARS):
        bwcn_nc = os.path.join(ROOT_DIR, f"O3_pc_BWCN_60_90N_{tag}_OctSep_year{y:04d}.nc")
        if not os.path.exists(bwcn_nc):
            print(f"[Plot] Missing BWCN series for year {y:04d}: {bwcn_nc}")
            continue

        bwcn_da = load_da(bwcn_nc)

        ax2.plot(
            x_full,
            bwcn_da.values,
            lw=1.5,
            color=colors_spec[i],
            label=f"Year {y:04d}",
        )

        handles_years.append(
            Line2D([0], [0], color=colors_spec[i], lw=1.5, label=f"Year {y:04d}")
        )

    ax2.fill_between(
        x_full,
        all_mean.values - all_std.values,
        all_mean.values + all_std.values,
        color="0.85",
        alpha=0.9,
        linewidth=0,
        zorder=0,
    )
    ax2.plot(
        x_full,
        all_mean.values,
        ls="--",
        lw=1.8,
        color="black",
        label="B2000 climatology (all years)",
    )

    ax2.fill_between(
        x_full,
        low25_mean.values - low25_std.values,
        low25_mean.values + low25_std.values,
        facecolor="none",
        edgecolor="0.5",
        hatch="///",
        linewidth=0.0,
        zorder=1,
    )
    ax2.plot(
        x_full,
        low25_mean.values,
        ls="-",
        lw=2.0,
        color="black",
        label="B2000 low-ozone composite (bottom 25%)",
    )

    ax2.set_xlim(0, N_DAYS - 1)
    ax2.set_xticks(month_ticks)
    ax2.set_xticklabels(month_labels)
    ax2.set_ylabel(f"Partial O$_3$ column, {int(ptop)}–{int(pbot)} hPa (DU)")
    ax2.set_title(f"Polar cap (60–90°N) partial ozone column ({int(ptop)}–{int(pbot)} hPa)")
    ax2.grid(False)

    patch_all = Patch(facecolor="0.85", edgecolor="none", label="B2000 all-year ±1σ")
    patch_low = Patch(facecolor="none", edgecolor="0.5", hatch="///", label="B2000 low-ozone ±1σ")
    line_all  = Line2D([0], [0], color="black", lw=1.8, ls="--", label="B2000 climatology")
    line_low  = Line2D([0], [0], color="black", lw=2.0, ls="-", label="B2000 low-ozone composite")

    handles = handles_years + [line_all, patch_all, line_low, patch_low]
    ax2.legend(handles=handles, loc="best", frameon=False, fontsize=8, ncol=2)

    out_png2 = os.path.join(ROOT_DIR, f"O3_daily_BWCN_vs_B2000_climatology_hybrid_{tag}.png")
    out_pdf2 = os.path.join(ROOT_DIR, f"O3_daily_BWCN_vs_B2000_climatology_hybrid_{tag}.pdf")

    plt.savefig(out_png2, dpi=300)
    plt.savefig(out_pdf2)
    plt.show()

    print("[Plot] Saved fig2:")
    print("   ", out_png2)
    print("   ", out_pdf2)

print("\n[Done] All plots generated successfully.")